In [1]:
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


In [2]:
# Load metrics
df = pd.read_csv("data/processed/menu_metrics.csv")

# Recreate thresholds (same as Day 3)
HIGH_SALES_THRESHOLD = df["units_sold_last_month"].quantile(0.6)
LOW_MARGIN_THRESHOLD = 50
LONG_PREP_THRESHOLD = 12


In [3]:
# Load FAISS
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.load_local(
    "data/processed/faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 10})


/var/folders/s1/s94qmf4129q34gqmyptfrk4w0000gn/T/ipykernel_33647/2181189000.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


In [4]:
docs = retriever.invoke("Which menu items are most profitable?")
[(d.metadata["item_name"], d.metadata["profit"]) for d in docs]


[('Mac and Cheese', 432.53),
 ('Nacho Platter', 4248.4),
 ('Bruschetta', 207.69),
 ('Mini Quesadillas', 1680.0),
 ('Loaded Fries', 2327.5),
 ('Seasonal Fruit Plate', 1003.2),
 ('Hot Chocolate', 836.0),
 ('Chicken Alfredo', 8874.36),
 ('Coleslaw', 1278.9),
 ('Side Salad', 669.24)]

In [5]:
def most_profitable(docs, top_n=3):
    return sorted(
        docs,
        key=lambda d: d.metadata["profit"],
        reverse=True
    )[:top_n]


In [10]:
def high_sales_low_margin(docs, top_n=3):
    strict = [
        d for d in docs
        if d.metadata["units_sold"] >= HIGH_SALES_THRESHOLD
        and d.metadata["profit_margin"] < LOW_MARGIN_THRESHOLD
    ]

    if strict:
        return strict[:top_n]

    # refined fallback: keep only bottom 40% margins among retrieved docs
    margin_cutoff = pd.Series(
        [d.metadata["profit_margin"] for d in docs]
    ).quantile(0.4)

    fallback = [
        d for d in docs
        if d.metadata["profit_margin"] <= margin_cutoff
    ]

    return sorted(
        fallback,
        key=lambda d: (-d.metadata["units_sold"], d.metadata["profit_margin"])
    )[:top_n]

In [11]:
def long_prep_items(docs, top_n=3):
    return sorted(
        docs,
        key=lambda d: d.metadata["prep_time"],
        reverse=True
    )[:top_n]


In [12]:
def answer_query(query):
    q = query.lower()
    docs = retriever.invoke(query)

    if "most profitable" in q:
        return most_profitable(docs)

    if "sell a lot" in q and "low margin" in q:
        return high_sales_low_margin(docs)

    if "long to prepare" in q or "take long" in q:
        return long_prep_items(docs)

    return docs[:3]


In [13]:
queries = [
    "Which menu items are most profitable?",
    "Which items sell a lot but have low margins?",
    "Items that take long to prepare"
]

for q in queries:
    print("\nQUERY:", q)
    docs = answer_query(q)
    for d in docs:
        print("-", d.metadata["item_name"], "|", d.metadata)



QUERY: Which menu items are most profitable?
- Chicken Alfredo | {'item_id': 204, 'item_name': 'Chicken Alfredo', 'category': 'Main_Courses', 'price': 22.0, 'units_sold': 498, 'profit_margin': 81, 'profit': 8874.36, 'prep_time': 13, 'menu_class': 'High-High', 'source': 'pos_sql'}
- Nacho Platter | {'item_id': 110, 'item_name': 'Nacho Platter', 'category': 'Appetizers', 'price': 13.0, 'units_sold': 430, 'profit_margin': 76, 'profit': 4248.4, 'prep_time': 9, 'menu_class': 'High-High', 'source': 'pos_sql'}
- Loaded Fries | {'item_id': 102, 'item_name': 'Loaded Fries', 'category': 'Appetizers', 'price': 12.5, 'units_sold': 490, 'profit_margin': 38, 'profit': 2327.5, 'prep_time': 8, 'menu_class': 'High-High', 'source': 'pos_sql'}

QUERY: Which items sell a lot but have low margins?
- Seasonal Fruit Plate | {'item_id': 302, 'item_name': 'Seasonal Fruit Plate', 'category': 'Desserts', 'price': 7.5, 'units_sold': 304, 'profit_margin': 44, 'profit': 1003.2, 'prep_time': 3, 'menu_class': 'Low-L